# Week 6: Single-cell RNA-seq Analysis

**Goal**: Learn how to perform simple single-cell RNA-seq analysis

**Reference**: [Single-cell Best Practices Book](https://www.sc-best-practices.org/introduction/raw_data_processing.html)

## Pipeline Overview

1. **Data Acquisition and Preprocessing** - Obtain FASTQ files, reference genome, GTF annotations
2. **Alevin-fry Pipeline** - Build index, alignment, quantification
3. **Cell Clustering** - Leiden algorithm clustering
4. **Cell Type Annotation** - Automatic annotation with CellTypist

---

## Step 1: Installation and Environment Setup

The following environment has been configured for this assignment:

```bash
# Tools installed in WSL
- conda (miniconda)
- salmon 1.10.3
- alevin-fry 0.11.2
- gffread 0.12.7

# Python packages
- scanpy, anndata, pyroe
- leidenalg, python-igraph
- celltypist, jupyter
```

Data file locations:
- R1/R2 reads: `data/toy_ref_read/toy_read_fastq/`
- Reference genome: `data/toy_ref_read/toy_human_ref/fasta/genome.fa`
- GTF annotations: `data/toy_ref_read/toy_human_ref/genes/genes.gtf`
- Whitelist: `data/3M-february-2018.txt.gz`


## Step 1: Download Data Files

Download required data files if they don't already exist locally. This step is automatically skipped if data is already present.


In [ ]:
# Download data files if they don't exist
import subprocess
import tarfile
import re
import os
from pathlib import Path

# Data URLs
BOX_DATA_URL = "https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814"
# Direct Box API download link (extracted from the Box page)
BOX_API_DOWNLOAD_URL = "https://public.boxcloud.com/api/2.0/files/964122990740/content"
WHITELIST_URL = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"

# Check if data directory exists
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Download and extract Box data if needed
toy_data_dir = data_dir / "toy_ref_read"
if not toy_data_dir.exists() or not list(toy_data_dir.glob("**/*")):
    print("Downloading data from Box...")
    box_archive = data_dir / "toy_read_ref_set.tar.gz"
    
    # Try to download from Box using direct API link first, then fallback methods
    try:
        # Priority 1: Use direct Box API download link (most reliable)
        download_urls_to_try = [
            BOX_API_DOWNLOAD_URL,  # Direct API link (highest priority)
        ]
        
        # Priority 2: Try to extract download URL from Box HTML page
        try:
            import urllib.request
            print("Attempting to extract download URL from Box page...")
            req = urllib.request.Request(
                BOX_DATA_URL,
                headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
            )
            
            with urllib.request.urlopen(req, timeout=30) as response:
                html_content = response.read().decode('utf-8', errors='ignore')
            
            # Try to find the download URL in the HTML
            patterns = [
                r'https://[^"]*boxcloud\.com[^"]*content[^"]*',
                r'https://[^"]*box\.com[^"]*download[^"]*',
                r'https://[^"]*box\.com/shared/static/[^"]*',
                r'"downloadUrl":"([^"]+)"',
                r'download_url["\']:\s*["\']([^"\']+)["\']',
            ]
            
            for pattern in patterns:
                matches = re.findall(pattern, html_content, re.IGNORECASE)
                if matches:
                    extracted_url = matches[0] if isinstance(matches[0], str) else matches[0]
                    if extracted_url not in download_urls_to_try:
                        download_urls_to_try.append(extracted_url)
                        print(f"Found additional download URL from HTML")
                        break
        except Exception as e:
            print(f"Could not extract URL from HTML (will try other methods): {e}")
        
        # Priority 3: Standard Box download URL formats
        download_urls_to_try.extend([
            f"{BOX_DATA_URL}?dl=1",
            f"{BOX_DATA_URL.replace('/s/', '/shared/static/')}?dl=1",
        ])
        
        download_success = False
        last_error = None
        
        for download_url in download_urls_to_try:
            print(f"Trying download URL: {download_url[:80]}...")
            try:
                # Try wget first
                try:
                    result = subprocess.run(
                        ["wget", "--no-check-certificate", "--content-disposition", 
                         "-O", str(box_archive), download_url],
                        capture_output=True,
                        text=True,
                        timeout=600
                    )
                    if result.returncode == 0 and box_archive.exists():
                        file_size = box_archive.stat().st_size
                        if file_size > 1000:
                            with open(box_archive, 'rb') as f:
                                magic = f.read(2)
                                if magic == b'\x1f\x8b':  # Valid gzip
                                    print(f"✓ Downloaded {box_archive.name} ({file_size:,} bytes) using wget")
                                    download_success = True
                                    break
                except FileNotFoundError:
                    pass
                
                # Fallback to curl
                result = subprocess.run(
                    ["curl", "-L", "-o", str(box_archive), download_url],
                    capture_output=True,
                    text=True,
                    timeout=600
                )
                if result.returncode == 0 and box_archive.exists():
                    file_size = box_archive.stat().st_size
                    if file_size > 1000:
                        with open(box_archive, 'rb') as f:
                            magic = f.read(2)
                            if magic == b'\x1f\x8b':  # Valid gzip
                                print(f"✓ Downloaded {box_archive.name} ({file_size:,} bytes) using curl")
                                download_success = True
                                break
            except Exception as e:
                last_error = e
                continue
        
        if not download_success:
            # Check if we're in CI environment
            is_ci = os.getenv('CI') == 'true' or os.getenv('GITHUB_ACTIONS') == 'true'
            
            error_msg = (
                f"\n{'='*60}\n"
                f"Box download failed - Manual download required\n"
                f"{'='*60}\n"
                f"Box shared links require JavaScript interaction and cannot be\n"
                f"automatically downloaded via command-line tools.\n\n"
                f"Please manually download the file:\n"
                f"  1. Visit: {BOX_DATA_URL}\n"
                f"  2. Click the 'Download' button\n"
                f"  3. Save as: {box_archive}\n"
                f"  4. Extract to: {toy_data_dir}\n\n"
            )
            
            if is_ci:
                error_msg += (
                    f"For CI environments, consider:\n"
                    f"  - Uploading the data to GitHub Releases\n"
                    f"  - Using a different file hosting service\n"
                    f"  - Pre-populating the data directory\n"
                )
            
            raise Exception(error_msg)
        
        # Verify archive was downloaded (not an HTML error page)
        file_size = box_archive.stat().st_size
        if file_size < 1000:
            raise ValueError(f"Downloaded file too small ({file_size} bytes), likely an error page")
        
        # Check if file is actually a gzip file (not HTML)
        with open(box_archive, 'rb') as f:
            magic_bytes = f.read(2)
            if magic_bytes != b'\x1f\x8b':  # Gzip magic number
                # Read first 100 bytes to check if it's HTML
                f.seek(0)
                first_bytes = f.read(100)
                if b'<html' in first_bytes.lower() or b'<!doctype' in first_bytes.lower():
                    raise ValueError(
                        f"Downloaded file appears to be HTML (not a tar.gz file). "
                        f"This usually means the Box link requires authentication or special handling.\n"
                        f"First 100 bytes: {first_bytes[:100]}\n"
                        f"Please manually download from: {BOX_DATA_URL}\n"
                        f"And extract to data/toy_ref_read/"
                    )
                else:
                    raise ValueError(
                        f"Downloaded file is not a valid gzip file. "
                        f"Magic bytes: {magic_bytes}, File size: {file_size} bytes"
                    )
        
        # Extract archive
        print("Extracting archive...")
        with tarfile.open(box_archive, "r:gz") as tar:
            tar.extractall(path=data_dir)
        print("✓ Archive extracted")
        
        # Clean up archive
        box_archive.unlink()
        print("✓ Cleaned up archive file")
    except Exception as e:
        print(f"Error downloading Box data: {e}")
        print("Note: Box links may require manual download. Please download from:")
        print(f"  {BOX_DATA_URL}")
        print("And extract to data/toy_ref_read/")
        raise
else:
    print("✓ Data directory already exists, skipping download")

# Download whitelist if needed
whitelist_file = data_dir / "3M-february-2018.txt.gz"
if not whitelist_file.exists():
    print("Downloading whitelist barcodes...")
    try:
        result = subprocess.run(
            ["wget", "-O", str(whitelist_file), WHITELIST_URL],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode != 0:
            # Fallback to curl
            result = subprocess.run(
                ["curl", "-L", "-o", str(whitelist_file), WHITELIST_URL],
                capture_output=True,
                text=True,
                timeout=300
            )
            if result.returncode != 0:
                raise subprocess.CalledProcessError(result.returncode, "curl", result.stderr)
        print(f"✓ Downloaded {whitelist_file.name}")
    except Exception as e:
        print(f"Error downloading whitelist: {e}")
        raise
else:
    print("✓ Whitelist file already exists")

print("\n✓ Data download complete!")


## Step 2: Import Libraries and Setup

Import all required Python libraries for the analysis.


In [ ]:
# Import required libraries
import os
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Define paths
BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = BASE_DIR / 'figures'
SCRIPTS_DIR = BASE_DIR / 'scripts'
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Define all input and output paths for reference
# Input paths
FASTQ_R1 = DATA_DIR / 'toy_ref_read' / 'toy_read_fastq' / 'selected_R1_reads.fastq'
FASTQ_R2 = DATA_DIR / 'toy_ref_read' / 'toy_read_fastq' / 'selected_R2_reads.fastq'
GENOME_FA = DATA_DIR / 'toy_ref_read' / 'toy_human_ref' / 'fasta' / 'genome.fa'
GTF_FILE = DATA_DIR / 'toy_ref_read' / 'toy_human_ref' / 'genes' / 'genes.gtf'
WHITELIST_FILE = DATA_DIR / '3M-february-2018.txt.gz'

# Output paths
TRANSCRIPTS_FA = RESULTS_DIR / 'transcripts.fa'
SALMON_INDEX = RESULTS_DIR / 'salmon_index_tx'
# Following best practices: separate directories for each pipeline stage
SALMON_ALEVIN_DIR = RESULTS_DIR / 'salmon_alevin'  # salmon alevin output (map.rad)
ALEVIN_FRY_GPL_DIR = RESULTS_DIR / 'alevin_fry_gpl'  # alevin-fry permit + collate
QUANT_DIR = RESULTS_DIR / 'af_quant_tx'  # final quantification output
T2G_FILE = RESULTS_DIR / 't2g_2col.tsv'
ADATA_RAW = RESULTS_DIR / 'adata_raw.h5ad'
ADATA_PROCESSED = RESULTS_DIR / 'adata_processed.h5ad'
ADATA_ANNOTATED = RESULTS_DIR / 'adata_annotated.h5ad'
UMAP_FIGURE = FIGURES_DIR / 'umap_clusters_and_celltypes.png'

print(f"Base directory: {BASE_DIR.absolute()}")
print(f"Data directory: {DATA_DIR.absolute()}")
print(f"Results directory: {RESULTS_DIR.absolute()}")
print(f"Figures directory: {FIGURES_DIR.absolute()}")
print(f"\nAll paths defined. Bash cells use relative paths matching these definitions.")


In [ ]:
# Verify all required paths exist
print("=== Verifying Input Paths ===")
input_paths = {
    "FASTQ R1": FASTQ_R1,
    "FASTQ R2": FASTQ_R2,
    "Genome FA": GENOME_FA,
    "GTF File": GTF_FILE,
    "Whitelist": WHITELIST_FILE,
}

for name, path in input_paths.items():
    if path.exists():
        print(f"✓ {name}: {path}")
    else:
        print(f"✗ {name}: {path} (MISSING)")

print("\n=== Output Directories ===")
output_dirs = [RESULTS_DIR, FIGURES_DIR, SCRIPTS_DIR]
for dir_path in output_dirs:
    if dir_path.exists():
        print(f"✓ {dir_path.name}/: {dir_path}")
    else:
        print(f"✗ {dir_path.name}/: {dir_path} (will be created)")

print("\n=== Path Summary ===")
print(f"All bash cells use relative paths matching these definitions:")
print(f"  Input: data/toy_ref_read/...")
print(f"  Output: results/...")
print(f"  Scripts: scripts/...")
print(f"  Figures: figures/...")


## Step 3: Build Transcriptome Index

Extract transcriptome sequences from genome using GTF annotations, then build salmon index.

**Note**: These steps are executed using bash commands. In practice, they should be run once during setup.


In [ ]:
%%bash
# Extract transcriptome from genome + GTF
if [ ! -f results/transcripts.fa ]; then
    gffread data/toy_ref_read/toy_human_ref/genes/genes.gtf \
        -g data/toy_ref_read/toy_human_ref/fasta/genome.fa \
        -w results/transcripts.fa
    echo "Extracted $(grep -c '>' results/transcripts.fa) transcripts"
else
    echo "Transcriptome already extracted: $(grep -c '>' results/transcripts.fa) transcripts"
fi


In [ ]:
%%bash
# Build salmon index
if [ ! -d results/salmon_index_tx ]; then
    salmon index \
        -t results/transcripts.fa \
        -i results/salmon_index_tx \
        -k 31
    echo "✓ Index built successfully"
else
    echo "✓ Index already exists"
fi


## Step 4: Map Reads with Salmon Alevin

Align single-cell reads to the transcriptome using salmon alevin with Chromium v3 chemistry.


In [ ]:
%%bash
# Run salmon alevin (output: map.rad files)
# Directory: salmon_alevin (separate from alevin-fry steps)
if [ ! -d results/salmon_alevin ]; then
    salmon alevin \
        -l ISR \
        -i results/salmon_index_tx \
        -1 data/toy_ref_read/toy_read_fastq/selected_R1_reads.fastq \
        -2 data/toy_ref_read/toy_read_fastq/selected_R2_reads.fastq \
        -o results/salmon_alevin \
        -p 4 \
        --chromiumV3 \
        --sketch
    echo "✓ Mapping completed"
else
    echo "✓ Mapping already complete"
fi


## Step 5: Alevin-fry Processing

Process the mapped reads using alevin-fry to generate cell-by-gene count matrix.

This includes:
1. Generate permit list (filter valid cell barcodes using knee-distance method)
2. Collate RAD file
3. Create transcript-to-gene mapping
4. Quantify gene expression


In [ ]:
%%bash
# Generate permit list (filter valid cell barcodes)
# Directory: alevin_fry_gpl (separate from salmon_alevin output)
mkdir -p results/alevin_fry_gpl
if [ ! -f results/alevin_fry_gpl/permit_freq.bin ]; then
    alevin-fry generate-permit-list \
        -d fw \
        -i results/salmon_alevin \
        -o results/alevin_fry_gpl \
        --knee-distance
    echo "✓ Permit list generated"
else
    echo "✓ Permit list already exists"
fi


In [ ]:
%%bash
# Collate RAD file (write output to the alevin_fry_gpl directory)
# -r: read from salmon_alevin (map.rad files)
# -i: input/output to alevin_fry_gpl (permit list + collated output)
if [ ! -f results/alevin_fry_gpl/map.collated.rad ]; then
    alevin-fry collate \
        -r results/salmon_alevin \
        -i results/alevin_fry_gpl \
        -t 4
    echo "✓ Collation completed"
else
    echo "✓ Collation already complete"
fi


### Create a clean two-column t2g mapping

`alevin-fry` (v0.11.2) expects a transcript-to-gene mapping with either two columns (transcript, gene) or three columns where the third entry indicates the splicing state (S/U). To avoid format issues we rebuild a strictly two-column file directly from the toy GTF by parsing transcript features and extracting transcript_id and gene_id pairs. The resulting file contains 271 transcript→gene pairs and will be used in the quantification step.


In [ ]:
# Build a 2-column t2g file (transcript_id <tab> gene_id) directly from GTF
import re

def extract_t2g_2col(gtf_file, output_file):
    """
    Extracts transcript_id and gene_id from a GTF file and writes them to a 2-column TSV.
    """
    pairs = []
    seen = set()
    
    with open(gtf_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split('\t')
            if len(parts) < 9:
                continue
            
            feature_type = parts[2]
            attributes = parts[8]
            
            if feature_type == "transcript":
                # Extract transcript_id and gene_id using regex
                transcript_id_match = re.search(r'transcript_id "([^"]+)"', attributes)
                gene_id_match = re.search(r'gene_id "([^"]+)"', attributes)
                
                if transcript_id_match and gene_id_match:
                    transcript_id = transcript_id_match.group(1)
                    gene_id = gene_id_match.group(1)
                    
                    # Avoid duplicates
                    if transcript_id not in seen:
                        seen.add(transcript_id)
                        pairs.append((transcript_id, gene_id))
    
    # Sort by transcript_id
    pairs.sort()
    
    # Write to file
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, 'w') as f:
        for transcript_id, gene_id in pairs:
            f.write(f"{transcript_id}\t{gene_id}\n")
    
    return len(pairs)

# Generate t2g file
count = extract_t2g_2col(GTF_FILE, T2G_FILE)
print(f"✓ Created t2g mapping with {count} entries")
print(f"  Output: {T2G_FILE}")
print("\nFirst 5 entries:")
with open(T2G_FILE, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"  {line.strip()}")


In [ ]:
# Quick sanity check of the two-column t2g file
with open(T2G_FILE, 'r') as f:
    lines = f.readlines()
    print(f"Total entries: {len(lines)}")
    print(f"\nFirst 5 entries:")
    for i, line in enumerate(lines[:5]):
        print(f"  {line.strip()}")


In [ ]:
%%bash
# Quantify gene expression using the two-column t2g mapping
# -i: read from alevin_fry_gpl (collated RAD files)
# -o: write to af_quant_tx (final expression matrix)
rm -rf results/af_quant_tx
alevin-fry quant \
    -r cr-like \
    -m results/t2g_2col.tsv \
    -i results/alevin_fry_gpl \
    -o results/af_quant_tx \
    -t 4 \
    --use-mtx

echo "\nQuantification outputs:"
ls -lh results/af_quant_tx/


## Step 6: Load Data into AnnData

Load the quantification results into an AnnData object for downstream analysis.


In [ ]:
# Load quantification data (manual MTX reading)
import scipy.io
quant_dir = QUANT_DIR / "alevin"
mtx_file = quant_dir / "quants_mat.mtx"
barcodes_file = quant_dir / "quants_mat_rows.txt"
features_file = quant_dir / "quants_mat_cols.txt"

X = scipy.io.mmread(mtx_file).tocsr()  # rows = cells, columns = genes
obs = pd.DataFrame(index=[line.strip() for line in open(barcodes_file)])
var = pd.DataFrame(index=[line.strip() for line in open(features_file)])
adata = ad.AnnData(X=X, obs=obs, var=var)

print(f"\n✓ Data loaded successfully!")
print(f"  Shape: {adata.shape}")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")


In [ ]:
# Calculate basic statistics
adata.var['n_cells'] = (adata.X > 0).sum(axis=0).A1
adata.obs['n_counts'] = adata.X.sum(axis=1).A1
adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1

print("\nBasic statistics:")
print(f"  Total UMI counts: {adata.X.sum():,.0f}")
print(f"  Mean counts per cell: {adata.obs['n_counts'].mean():,.2f}")
print(f"  Median counts per cell: {adata.obs['n_counts'].median():,.2f}")
print(f"  Mean genes per cell: {adata.obs['n_genes'].mean():,.2f}")

# Save raw AnnData
adata.write_h5ad("results/adata_raw.h5ad")
print(f"\n✓ Saved to results/adata_raw.h5ad")


## Step 7: Normalize, reduce dimensionality, and cluster (Leiden)

We apply a simple Scanpy workflow: total-count normalization, log transform, highly variable gene selection, PCA, neighbor graph construction, UMAP embedding, and Leiden clustering. The processed AnnData is saved for downstream visualization and annotation.


In [ ]:
# Basic normalization and clustering pipeline
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=min(adata.n_vars, 20), flavor="seurat")
if adata.var['highly_variable'].sum() > 0:
    adata = adata[:, adata.var['highly_variable']]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=min(adata.n_vars, 20))
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

print(f"Processed shape: {adata.shape}")
print("Leiden clusters:", adata.obs['leiden'].value_counts().to_dict())

adata.write_h5ad("results/adata_processed.h5ad")
print("✓ Saved processed AnnData to results/adata_processed.h5ad")


: 

### Visualize Clustering Results

Plot the UMAP embedding colored by Leiden clusters to visualize the cell structure identified by the clustering algorithm.


In [ ]:
# Plot UMAP with Leiden clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Leiden Clustering', 
           save='_leiden_clusters.png', show=True)

# Print cluster summary
print("\nCluster Summary:")
print(adata.obs['leiden'].value_counts().sort_index())


## Step 8: Cell Type Annotation with CellTypist

Perform automatic cell type annotation using CellTypist. CellTypist uses pre-trained models to predict cell types based on gene expression patterns.


In [ ]:
# Import CellTypist
import celltypist
from celltypist import models

# Note: CellTypist requires gene symbols (e.g., GAPDH, ACTB), but our toy dataset
# uses ENSEMBL IDs (e.g., ENSG00000113575) and only has 20 genes.
# This limits CellTypist's ability to match features with pre-trained models.
# For demonstration, we'll attempt annotation but expect limited results.

print("Preparing CellTypist annotation...")
print("Note: Toy dataset has ENSEMBL IDs and only 20 genes, which may limit CellTypist predictions.")


In [ ]:
# Step 1: Read GTF and build gene_id → gene_name mapping
import scipy.io

gene_map = {}
gtf_file = Path("data/toy_ref_read/toy_human_ref/genes/genes.gtf")

with open(gtf_file, 'r') as f:
    for line in f:
        if "\tgene\t" in line and "gene_name" in line:
            # Extract gene_id
            if 'gene_id "' in line:
                gene_id = line.split('gene_id "')[1].split('"')[0]
                # Extract gene_name
                if 'gene_name "' in line:
                    gene_name = line.split('gene_name "')[1].split('"')[0]
                    gene_map[gene_id] = gene_name

gene_map = pd.Series(gene_map)
print(f"✓ Built gene mapping: {len(gene_map)} genes")

# Step 2: Reload raw data and apply mapping to convert ENSEMBL IDs to gene symbols
quant_dir = QUANT_DIR / "alevin"
X_raw = scipy.io.mmread(quant_dir / "quants_mat.mtx").tocsr()
obs_raw = pd.DataFrame(index=[line.strip() for line in open(quant_dir / "quants_mat_rows.txt")])
var_raw = pd.DataFrame(index=[line.strip() for line in open(quant_dir / "quants_mat_cols.txt")])
adata_for_ct = ad.AnnData(X=X_raw, obs=obs_raw, var=var_raw)

# Apply gene mapping
adata_for_ct.var["gene_symbols"] = adata_for_ct.var_names.map(gene_map)
adata_for_ct = adata_for_ct[:, adata_for_ct.var["gene_symbols"].notnull()].copy()
adata_for_ct.var_names = adata_for_ct.var["gene_symbols"]
adata_for_ct.var_names_make_unique()

print(f"✓ Converted to gene symbols: {adata_for_ct.shape}")
print(f"  Genes with symbols: {adata_for_ct.n_vars}")

# Normalize for CellTypist (needs log1p normalized data)
sc.pp.normalize_total(adata_for_ct, target_sum=1e4)
sc.pp.log1p(adata_for_ct)

# Pre-compute PCA for majority_voting (which needs neighbor graph)
# PCA requires n_comps < min(n_samples, n_features)
n_pcs = min(adata_for_ct.n_vars - 1, adata_for_ct.n_obs - 1, 50)
if n_pcs > 0:
    sc.tl.pca(adata_for_ct, n_comps=n_pcs, svd_solver='arpack')
    sc.pp.neighbors(adata_for_ct, n_neighbors=min(10, adata_for_ct.n_obs-1), n_pcs=n_pcs)
else:
    print("Warning: Cannot compute PCA, majority_voting may be limited")

# Step 3: Run CellTypist
print("\nRunning CellTypist annotation...")
pred = celltypist.annotate(adata_for_ct, model='Immune_All_Low.pkl', majority_voting=True)
adata_ct = pred.to_adata()

# Add predictions to processed adata (which has UMAP coordinates)
adata.obs['cell_type'] = adata_ct.obs['predicted_labels']
if 'majority_voting' in adata_ct.obs.columns:
    adata.obs['cell_type'] = adata_ct.obs['majority_voting']
if 'confidence_score' in adata_ct.obs.columns:
    adata.obs['cell_type_score'] = adata_ct.obs['confidence_score']

print("\n✓ Cell type annotation completed!")
print("\nPredicted cell types:")
print(adata.obs['cell_type'].value_counts())

# Step 4: Save annotated data
adata.write_h5ad("results/adata_annotated.h5ad")
print("\n✓ Saved annotated AnnData to results/adata_annotated.h5ad")


### Visualize Cell Type Annotations

Plot the UMAP embedding colored by predicted cell types from CellTypist.


In [ ]:
# Plot UMAP with cell type annotations
sc.pl.umap(adata, color='cell_type', legend_loc='right margin', 
           title='Cell Type Annotation (CellTypist)', 
           save='_celltypes.png', show=True)

# Also show both leiden clusters and cell types side by side
Path('figures').mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Leiden Clusters')
sc.pl.umap(adata, color='cell_type', ax=axes[1], show=False, title='Cell Types (CellTypist)', 
           legend_loc='right margin')
plt.tight_layout()
plt.savefig('figures/umap_clusters_and_celltypes.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to figures/umap_clusters_and_celltypes.png")


## Summary and Time Log

### Pipeline Completion Summary

This notebook successfully implements the complete single-cell RNA-seq analysis pipeline:

1. ✅ **Data Acquisition and Preprocessing** - Extracted transcriptome from genome and GTF
2. ✅ **Alevin-fry Pipeline** - Built index, aligned reads, generated permit list, collated RAD files, and quantified gene expression
3. ✅ **Cell Clustering** - Applied Leiden modularity algorithm and visualized clusters
4. ✅ **Cell Type Annotation** - Used CellTypist to automatically annotate cell types

### Key Results

- **Cells analyzed**: 114 cells
- **Genes detected**: 20 genes (toy dataset)
- **Leiden clusters**: 2 clusters (57 cells each)
- **Cell types identified**: 4 types (Epithelial cells: 89, Tcm/Naive helper T cells: 18, Double-positive thymocytes: 4, Fibroblasts: 3)

### Time Log

**Total time spent**: ~6-8 hours

Breakdown:
- Environment setup and tool installation: ~1 hour
- Alevin-fry pipeline debugging and optimization: ~4 hours
  - Quantification debugging (t2g format issues): ~3 hours
- Clustering and visualization: ~1 hour
- CellTypist annotation setup (gene symbol conversion): ~1 hour
- Documentation and notebook organization: ~30 minutes
- CI and github submition: ~1 hour

**Note**: The majority of time was spent debugging the alevin-fry quantification step, particularly resolving the t2g file format issue (2-column vs 3-column format required by alevin-fry 0.11.2).
